In [1]:
# retrain_full_adder_robust.py
import numpy as np
import joblib
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report

# Truth table (A, B, Cin)
X = np.array([
    [0,0,0],
    [0,0,1],
    [0,1,0],
    [0,1,1],
    [1,0,0],
    [1,0,1],
    [1,1,0],
    [1,1,1]
], dtype=float)

y_sum = np.array([0,1,1,0,1,0,0,1])
y_carry = np.array([0,0,0,1,0,1,1,1])

# Candidate hyperparams to try
solvers = ['lbfgs', 'adam']
hidden_sizes = [(4,), (6,), (8,)]
random_states = [1, 7, 13, 42, 99]

def find_best_model(y_target, task_name):
    best = None
    best_acc = -1.0
    best_info = None

    for solver in solvers:
        for hs in hidden_sizes:
            for rs in random_states:
                # For lbfgs, loss_curve_ is not available; it's okay.
                clf = MLPClassifier(hidden_layer_sizes=hs,
                                    activation='logistic' if solver=='lbfgs' else 'relu',
                                    solver=solver,
                                    learning_rate_init=0.01 if solver=='adam' else 0.05,
                                    max_iter=5000 if solver=='lbfgs' else 2000,
                                    random_state=rs)
                try:
                    clf.fit(X, y_target)
                except Exception as e:
                    # skip failed trainings
                    print(f"[skip] solver={solver}, hs={hs}, rs={rs} -> error: {e}")
                    continue

                preds = clf.predict(X).astype(int)
                acc = accuracy_score(y_target, preds)

                # Save if perfect or better than best
                if acc > best_acc:
                    best = clf
                    best_acc = acc
                    best_info = (solver, hs, rs, preds)

                print(f"[{task_name}] solver={solver}, hs={hs}, rs={rs} -> acc={acc:.3f}")

                if acc == 1.0:
                    print(f"[{task_name}] Found perfect model: solver={solver}, hs={hs}, rs={rs}")
                    return best, best_info

    return best, best_info

# Find best SUM model
sum_model, sum_info = find_best_model(y_sum, "SUM")
# Find best CARRY model
carry_model, carry_info = find_best_model(y_carry, "CARRY")

if sum_model is None or carry_model is None:
    raise RuntimeError("Failed to train models; check environment and sklearn version.")

# Evaluate and print verification table
pred_sum = sum_model.predict(X).astype(int)
pred_carry = carry_model.predict(X).astype(int)

print("\n=== Verification ===")
print(" A B Cin | SUM_pred CARRY_pred | exp_SUM exp_CARRY")
print("--------------------------------------------------")
for i, x in enumerate(X.astype(int)):
    print(f" {x[0]} {x[1]}  {x[2]}  |   {pred_sum[i]}        {pred_carry[i]}    |    {y_sum[i]}      {y_carry[i]}")

print("\nSUM model info (solver, hs, rs):", sum_info[:3])
print("CARRY model info (solver, hs, rs):", carry_info[:3])

print("\nSUM accuracy:", accuracy_score(y_sum, pred_sum))
print(classification_report(y_sum, pred_sum, zero_division=0))
print("\nCARRY accuracy:", accuracy_score(y_carry, pred_carry))
print(classification_report(y_carry, pred_carry, zero_division=0))

[SUM] solver=lbfgs, hs=(4,), rs=1 -> acc=0.500
[SUM] solver=lbfgs, hs=(4,), rs=7 -> acc=1.000
[SUM] Found perfect model: solver=lbfgs, hs=(4,), rs=7
[CARRY] solver=lbfgs, hs=(4,), rs=1 -> acc=1.000
[CARRY] Found perfect model: solver=lbfgs, hs=(4,), rs=1

=== Verification ===
 A B Cin | SUM_pred CARRY_pred | exp_SUM exp_CARRY
--------------------------------------------------
 0 0  0  |   0        0    |    0      0
 0 0  1  |   1        0    |    1      0
 0 1  0  |   1        0    |    1      0
 0 1  1  |   0        1    |    0      1
 1 0  0  |   1        0    |    1      0
 1 0  1  |   0        1    |    0      1
 1 1  0  |   0        1    |    0      1
 1 1  1  |   1        1    |    1      1

SUM model info (solver, hs, rs): ('lbfgs', (4,), 7)
CARRY model info (solver, hs, rs): ('lbfgs', (4,), 1)

SUM accuracy: 1.0
              precision    recall  f1-score   support

           0       1.00      1.00      1.00         4
           1       1.00      1.00      1.00         4

   

In [3]:
import joblib
joblib.dump(sum_model, "frontend/models/full_adder_sum_mlp.pkl")

['frontend/models/full_adder_sum_mlp.pkl']

In [4]:
joblib.dump(carry_model, "frontend/models/full_adder_carry_mlp.pkl")

['frontend/models/full_adder_carry_mlp.pkl']